# Notebook A: Train LoRA Qwen3.5-4B (Unsloth)
**Thứ tự chạy:**
1. Chạy cell 1 (kiểm tra GPU) và cell 2 (cài đặt) **một lần**.
2. **Runtime → Khởi động lại phiên** (Ctrl+M .).
3. Từ cell 3 trở đi chạy bình thường. **Không chạy lại cell 2.**

Adapter lưu vào Google Drive: `qwen_norm_4b_lora` (Tổng tham số hệ thống: 4B Norm + 9B Solver = 13B <= 15B chuẩn Rule 2). Train xong thì sang Notebook B để serve bằng vLLM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# CÀI ĐẶT (chạy 1 lần, xong phải Khởi động lại phiên)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade "transformers>=5.2.0"
!pip install -q --no-deps trl peft accelerate bitsandbytes
print("XONG. Bây giờ: Runtime -> Khởi động lại phiên, rồi chạy từ cell 3.")

In [ ]:
# Mount Drive + đường dẫn
from google.colab import drive
import os
drive.mount("/content/drive")

BASE = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
OUTPUT_LORA_DIR = f"{BASE}/qwen_norm_4b_v2_lora"

# File dữ liệu: ưu tiên /content, nếu không có thì tìm trên Drive
DATA_PATH = "/content/full_gold_norm_v2_chatml_cleaned.jsonl"
if not os.path.exists(DATA_PATH):
    DATA_PATH = f"{BASE}/full_gold_norm_v2_chatml_cleaned.jsonl"

print("DATA_PATH       =", DATA_PATH, "| tồn tại:", os.path.exists(DATA_PATH))
print("OUTPUT_LORA_DIR =", OUTPUT_LORA_DIR)

In [ ]:
# Kiểm tra phiên bản trước khi train
import numpy, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, "transformers phải >= 5.2.0, hãy restart session sau khi cài"
from unsloth import FastLanguageModel
print("OK")

In [ ]:
# TRAIN LoRA
import os
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

adapter_config = os.path.join(OUTPUT_LORA_DIR, "adapter_config.json")
if os.path.exists(adapter_config):
    print("ĐÃ CÓ ADAPTER tại", OUTPUT_LORA_DIR, "-> bỏ qua train. Muốn train lại thì xóa thư mục đó trên Drive.")
else:
    assert os.path.exists(DATA_PATH), f"Không thấy {DATA_PATH}"
    os.makedirs(OUTPUT_LORA_DIR, exist_ok=True)

    max_seq_length = 768
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="Qwen/Qwen3.5-4B",
        max_seq_length=max_seq_length,
        load_in_4bit=False,   # A100/L4: train bf16 cho khớp vLLM (--dtype bfloat16)
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=2026,
    )

    dataset = load_dataset("json", data_files=DATA_PATH, split="train")
    dataset = dataset.map(lambda ex: {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)})
    print("=== MẪU 1 SAU KHI ÁP TEMPLATE (kiểm tra format) ===")
    print(dataset[0]["text"])
    print("===================================================")

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text",
            max_length=max_seq_length,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            warmup_steps=30,
            num_train_epochs=3,
            learning_rate=2e-4,
            bf16=True,
            logging_steps=25,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=2026,
            output_dir="outputs",
            report_to="none",
        ),
    )
    trainer.train()

    model.save_pretrained(OUTPUT_LORA_DIR)
    tokenizer.save_pretrained(OUTPUT_LORA_DIR)
    print("ĐÃ LƯU ADAPTER:", OUTPUT_LORA_DIR)

In [ ]:
# Test nhanh adapter vừa train (cả câu thường và câu có Luật liên quan)
FastLanguageModel.for_inference(model)
NORM_SYSTEM_V2 = (
    "Bạn là chuyên gia chuẩn hóa câu hỏi tiếng Việt cho bài toán Text-to-SQL quản lý bán hàng.\n"
    "Nhiệm vụ: Sửa lỗi chính tả, teencode, viết tắt, thêm dấu và chuẩn hóa câu hỏi rõ ràng về hình thức và kiểu trả về (bảng, danh sách, %).\n"
    "Nếu có mục \"Luật liên quan:\", PHẢI giữ nguyên đúng tên thuật ngữ trong luật, không tự ý đổi sang chỉ số khác và không tự chèn giải thích dài trong ngoặc.\n"
    "TUYỆT ĐỐI GIỮ NGUYÊN các thẻ placeholder <ENT_X>. Chỉ trả về đúng 1 câu chuẩn hóa."
)
for sample_u in [
    # 1. Telex + viết tắt + so sánh %
    "Câu hỏi: thangf truwocs tiemj <ENT_0> dthu tang hay giam bn % so vs cung ky?",
    # 2. Từ địa phương nặng + nhiều <ENT_X> + xuất bảng
    "Câu hỏi: bữa ni coi giùm tui thợ mô làm nhìu lựơt xe nhất trong <ENT_0> ở chi nhánh <ENT_1>, xuất ra cái bảng xếp từ cao xuống thấp nghe?",
    # 3. Giữ đúng từ khóa luật dễ nhầm: Khách sộp nhất
    "Luật liên quan:\n| Khách sộp nhất | khách có **giá trị trung bình một đơn** cao nhất trong kỳ, chỉ xét khách có **từ 3 đơn trở lên** trong kỳ |\n\nCâu hỏi: <ENT_0> ai la khach sop nhat z?",
    # 4. Giữ đúng từ khóa luật dễ đảo nghĩa + phương ngữ 'chi': Hàng ế nhất
    "Luật liên quan:\n| Hàng ế nhất | của kỳ là mặt hàng có số lượng quy về đơn vị cơ bản thấp nhất trong các mặt hàng có bán trong kỳ |\n\nCâu hỏi: cho hoi mat hang e nhat <ENT_0> la mon chi?",
    # 5. Giữ đúng từ khóa luật dễ bịa: Đơn đa dạng
    "Luật liên quan:\n| Đơn đa dạng | là đơn §1 có từ 3 mặt hàng khác nhau trở lên |\n\nCâu hỏi: th<ENT_0> tiem co bn don da dang?",
    # 6. Ngành mới (trụ sạc kWh) + câu hỏi Boolean Có/Không
    "Câu hỏi: tu <ENT_0> toi <ENT_1>, tru sac <ENT_2> co san luong kWh thang sau deu cao hon thang trc ko ha?",
]:
    msgs = [{"role": "system", "content": NORM_SYSTEM_V2}, {"role": "user", "content": sample_u}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text=text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200)
    print("IN :", sample_u.splitlines()[-1])
    print("OUT:", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))